# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [2]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews.txt
Uploaded file name: product_reviews.txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [3]:
# Write your Answer here (Q1)
import pandas as pd
import numpy as np
import re

df = pd.read_csv('product_reviews.txt', sep='\t')
print("(a) df.shape:", df.shape)
print("(b) df.head(3):")
print(df.head(3))

(a) df.shape: (10, 2)
(b) df.head(3):
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [4]:
# Write your Answer here (Q2)
df['word_count']   = df['text'].apply(lambda x: len(x.split()))
df['char_count']   = df['text'].apply(len)
df['avg_word_len'] = df['text'].apply(
    lambda x: round(np.mean([len(w) for w in re.findall(r"[a-zA-Z']+", x)]), 4)
)
df['excl_count']   = df['text'].apply(lambda x: x.count('!'))

print(df[['id', 'word_count', 'char_count', 'avg_word_len', 'excl_count']].to_string(index=False))


 id  word_count  char_count  avg_word_len  excl_count
  1           9          55        5.0000           1
  2           7          51        6.3333           3
  3           8          45        4.5000           0
  4          10          56        4.5000           0
  5           8          53        5.5000           1
  6          10          51        4.0000           0
  7           9          50        4.4444           0
  8           6          40        5.5000           0
  9           7          52        6.2857           0
 10           7          48        4.7500           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [5]:
# # Write your Answer here (Q3)
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
X_cv = cv.fit_transform(df['text'])

vocab_size = len(cv.vocabulary_)
print("1) Vocabulary size:", vocab_size)

word_counts = np.asarray(X_cv.sum(axis=0)).flatten()
words = cv.get_feature_names_out()
top10_idx = word_counts.argsort()[::-1][:10]
print("\n2) Top 10 words by total count:")
for i in top10_idx:
    print(f"   {words[i]}: {word_counts[i]}")


1) Vocabulary size: 50

2) Top 10 words by total count:
   wouldn: 1
   works: 1
   working: 1
   want: 1
   value: 1
   twice: 1
   thanks: 1
   terrible: 1
   support: 1
   super: 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [6]:
# Write your Answer here (Q4)
cv2 = CountVectorizer(stop_words='english', ngram_range=(2, 2))
X_bi = cv2.fit_transform(df['text'])

bigram_counts = np.asarray(X_bi.sum(axis=0)).flatten()
bigrams = cv2.get_feature_names_out()
top5_bi = bigram_counts.argsort()[::-1][:5]
print("Top 5 bigrams by total count:")
for i in top5_bi:
    print(f"   {bigrams[i]}: {bigram_counts[i]}")

Top 5 bigrams by total count:
   wouldn buy: 1
   works fine: 1
   working days: 1
   want refund: 1
   value price: 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [7]:
# Write your Answer here  (Q5)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df['text'])

avg_tfidf = np.asarray(X_tfidf.mean(axis=0)).flatten()
terms = tfidf.get_feature_names_out()
top5_tf = avg_tfidf.argsort()[::-1][:5]
print("Top 5 terms by average TF-IDF score:")
for i in top5_tf:
    print(f"   {terms[i]}: {round(avg_tfidf[i], 4)}")


Top 5 terms by average TF-IDF score:
   setup: 0.0378
   setup easy: 0.0378
   instructions: 0.0378
   easy: 0.0378
   clear: 0.0378


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
